# Quantum Machine Learning from primitives

A quantum machine learning model is a parameterized quantum circuit whose parameters are tuned to fit data. This tutorial builds one from the ground up out of the primitives in `qrisp.qml`, trains it, and then varies each part in turn.

There is no model class to instantiate and no training loop to inherit. A model here is an ordinary Python function, the loss is an ordinary Python function, and training is a call to `qrisp.jasp.minimize`. Everything is assembled explicitly, which means every part can be replaced.

The pipeline has four stages:

```
   data  ->  ENCODING  ->  ANSATZ  ->  MEASUREMENT  ->  prediction
             (a state)   (parameters)  (a number)
                              ^                            |
                              |                            v
                              +-------- OPTIMIZER <----- LOSS
```

**Encoding** turns a feature vector into a quantum state. **Ansatz** applies a parameterized circuit to that state. **Measurement** reads out an expectation value, which is the model's prediction. A **loss** compares predictions to targets, and an **optimizer** adjusts the parameters to reduce it.

## The problem

We use a small binary classification task. Each sample has two features and a label of $+1$ or $-1$.

The features are rotation angles, so they must lie in an interval of length $2\pi$. Rotation angles are $2\pi$-periodic: a feature of $0.1$ and a feature of $2\pi + 0.1$ encode to the *same state*, and the model cannot tell them apart. `minmax_scale` puts raw features into a usable range, and we use it below.

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from qrisp import QuantumFloat
from qrisp.operators import X, Z
from qrisp.jasp import jaspify, minimize
from qrisp.qml import (
    Ansatz,
    Encoding,
    binarize,
    expectation,
    hardware_efficient_size,
    minmax_scale,
    normalize,
    real_amplitudes_size,
)

# Raw measurements on their own scales: nothing here is a rotation angle.
X_raw = np.array(
    [
        [12.0, 480.0],
        [15.0, 495.0],
        [38.0, 610.0],
        [34.0, 598.0],
    ]
)
y_train = jnp.array([-1.0, -1.0, 1.0, 1.0])

## Scaling

Feature 2 spans roughly 480 to 610. Used directly as a rotation angle that is about 20 full turns, so samples that are far apart in the data end up at nearly the same angle. Nothing raises an error — the circuit runs and the model simply fails to learn.

`minmax_scale` maps each feature from its range on the *training* data onto $[0, \pi]$ by default. The minimum and maximum come from the training set alone and the same two numbers are applied to any test set, so no information about the test data reaches the model during training.

In [ ]:
X_train = minmax_scale(X_raw)

fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3.6))

colors = ["tab:blue" if label < 0 else "tab:red" for label in np.asarray(y_train)]

left.scatter(X_raw[:, 0], X_raw[:, 1], c=colors, s=70)
left.set_title("raw features")
left.set_xlabel("feature 0")
left.set_ylabel("feature 1")

right.scatter(X_train[:, 0], X_train[:, 1], c=colors, s=70)
right.set_title("scaled to $[0, \\pi]$")
right.set_xlabel("feature 0")
right.set_ylabel("feature 1")
for axis in (right.set_xlim, right.set_ylim):
    axis(-0.2, np.pi + 0.2)

fig.tight_layout()
plt.show()

print("raw    range per feature:", X_raw.min(axis=0), "to", X_raw.max(axis=0))
print("scaled range per feature:", np.round(X_train.min(axis=0), 3), "to", np.round(X_train.max(axis=0), 3))

# The scaled features go on to the quantum side, so hand them over as a jax array.
X_train = jnp.asarray(X_train)

The shape of the data is unchanged — only the axes are. That is the point: the classes stay as separable as they were, but every value is now a usable rotation angle.

Two further helpers cover the other encodings' input requirements. `normalize` scales each sample to unit norm, which is what `Encoding.amplitude` needs, and `binarize` reduces features to the 0/1 values `Encoding.basis` works in.

In [ ]:
print("normalize :", np.round(normalize(np.array([[3.0, 4.0]])), 3), " (norm 5 -> 1)")
print("binarize  :", binarize(np.array([[0.8, -0.2, 1.5]])))

## Encoding

`Encoding.angle` writes each feature into the rotation angle of one qubit. A register of $n$ qubits therefore holds $n$ features.

For the default `axis="y"` it prepares the product state

$$\bigotimes_{i} \left( \cos(x_i/2)\ket{0} + \sin(x_i/2)\ket{1} \right).$$

A feature of $0$ leaves its qubit in $\ket{0}$; a feature of $\pi$ flips it to $\ket{1}$. The two clusters in our data therefore sit at opposite corners of the register.

In [ ]:
encoding = Encoding.angle(n_features=2)

qf = QuantumFloat(2)
encoding.apply(qf, [0.0, 0.0])
print("features [0, 0]   ->", qf.qs.statevector())

qf = QuantumFloat(2)
encoding.apply(qf, [np.pi, np.pi])
print("features [pi, pi] ->", qf.qs.statevector())

## Ansatz

The encoding is fixed by the data; the ansatz is what the optimizer controls. `Ansatz.real_amplitudes` alternates layers of $R_Y$ rotations with CX entanglers:

$$\left( \prod_{l=1}^{L} U_{\text{ent}} R_Y(w_l) \right) R_Y(w_0)$$

There is one more rotation layer than entangling layers, so the output of the last entangler is rotated before it is measured.

Every ansatz has a companion size function giving the number of parameters it needs. Use it rather than working the count out by hand — it keeps the parameter vector correct when the register size or layer count changes.

In [ ]:
num_qubits = 2
layers = 1
num_params = real_amplitudes_size(num_qubits, layers=layers)

print(f"{num_qubits} qubits, {layers} layer(s) ->  {num_params} parameters")

## Measurement

`expectation` turns a state preparation into a function returning an expectation value. It takes a **callable** that prepares the state, not a state — because estimating an expectation value requires running the circuit many times, so the preparation has to be repeatable.

The observable decides what is being read out. $Z(0)$ measures the first qubit along the Z axis, giving $+1$ for $\ket{0}$ and $-1$ for $\ket{1}$ — which matches the $\pm 1$ labels of a binary classifier.

In [ ]:
# Encoding followed by ansatz: this is the model.
encoding = Encoding.angle(n_features=num_qubits)
ansatz = Ansatz.real_amplitudes(num_qubits=num_qubits, layers=layers)


def circuit(x, w):
    qf = QuantumFloat(num_qubits)
    encoding.apply(qf, x)
    ansatz.apply(qf, w)
    return qf


# The model's prediction for one sample.
def model(x, w):
    return expectation(lambda: circuit(x, w), Z(0), precision=0.05)()

`precision` sets how tightly each expectation value is estimated. The shot count scales with the inverse square of it, so `0.05` is far cheaper than the default `0.01` and accurate enough while iterating.

## Loss and training

The loss is the mean squared error between predictions and labels. It takes only the parameters, which is the signature `minimize` expects.

`minimize` is **gradient-free**. It samples the loss rather than differentiating it, because the expectation values come from sampling and carry no derivative — tracing a circuit with Jasp makes it fast, but it does not make it differentiable.

In [ ]:
def make_loss(model_fn, X, y):
    def loss(w):
        total = 0.0
        for i in range(X.shape[0]):
            total = total + jnp.sum((model_fn(X[i], w) - y[i]) ** 2)
        return total / X.shape[0]

    return loss

Training runs inside `@jaspify`. Jasp owns the outermost layer of a hybrid program: quantum and classical code compose freely *inside* it, but it cannot be nested inside `jax.jit` from the outside.

Both solvers are **local**: they descend from the starting point into the nearest minimum. A run that stalls at a poor loss is not evidence that the model cannot fit the data — it usually means the starting point was bad. Training from several starts and keeping the best is the normal way to use them, and the cell below does exactly that.

In [ ]:
@jaspify(terminal_sampling=True)
def train():
    loss = make_loss(model, X_train, y_train)

    results = []
    for start in [0.5, 1.5, 2.5]:
        w0 = jnp.array([start] * num_params)
        result = minimize(loss, w0, method="COBYLA", options={"maxiter": 80})
        results.append((start, loss(w0), result.fun, result.x))
    return results


results = train()

for start, before, after, _ in results:
    print(f"start {start}:  loss {float(before):.3f} -> {float(after):.3f}")

best = min(results, key=lambda r: r[2])
print(f"\nbest final loss: {float(best[2]):.4f}")

The spread across starting points is the local-minimum effect: some starts reach a good fit and others stall well above it. Only the best is a statement about the model.

## Checking the predictions

A low loss is worth confirming against the predictions themselves. For a $\pm 1$ classifier the sign is what matters.

In [ ]:
w_opt = best[3]


@jaspify(terminal_sampling=True)
def predict_all():
    return jnp.array([model(X_train[i], w_opt) for i in range(X_train.shape[0])])


predictions = predict_all()

print("target      :", np.asarray(y_train))
print("prediction  :", np.round(np.asarray(predictions), 3))
print("signs match :", bool(np.all(np.sign(np.asarray(predictions)) == np.sign(np.asarray(y_train)))))

### Watching the loss fall

`minimize` returns only the final result, so to see the loss *during* training we run a short optimization by hand and record it at each step. This is a plain coordinate descent — deliberately simple, since the point is the curve rather than the method.

In [ ]:
@jaspify(terminal_sampling=True)
def train_recording():
    loss = make_loss(model, X_train, y_train)

    w = jnp.array([2.5] * num_params)
    history = [loss(w)]

    step = 0.3
    for _ in range(12):
        for i in range(num_params):
            here = loss(w)
            forward = loss(w.at[i].add(step))
            # Move the parameter whichever way the loss actually decreases.
            w = jnp.where(forward < here, w.at[i].add(step), w.at[i].add(-step))
        history.append(loss(w))

    return history


history = [float(value) for value in train_recording()]

plt.figure(figsize=(6, 3.6))
plt.plot(history, marker="o")
plt.xlabel("sweep")
plt.ylabel("loss")
plt.title("loss during training")
plt.grid(alpha=0.3)
plt.show()

print(f"loss {history[0]:.3f} -> {history[-1]:.3f}")

### The decision boundary

Evaluating the trained model on a grid of feature values shows what it learned across the whole input space, not just at the four training points.

In [ ]:
resolution = 12
grid_axis = np.linspace(0.0, np.pi, resolution)
grid = jnp.array([[a, b] for b in grid_axis for a in grid_axis])


@jaspify(terminal_sampling=True)
def predict_grid():
    return jnp.array([model(grid[i], w_opt) for i in range(grid.shape[0])])


surface = np.asarray(predict_grid()).reshape(resolution, resolution)

plt.figure(figsize=(5.2, 4.2))
plt.contourf(grid_axis, grid_axis, surface, levels=20, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="model output")
plt.contour(grid_axis, grid_axis, surface, levels=[0.0], colors="black", linewidths=2)

training = np.asarray(X_train)
plt.scatter(training[:, 0], training[:, 1], c=colors, s=90, edgecolors="black", zorder=3)
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.title("decision boundary")
plt.tight_layout()
plt.show()

The black line is where the model output crosses zero, which is where its prediction changes class. The training points sit on the sides matching their labels.

## Swapping the parts

Because the pipeline is assembled from separate objects, each stage can be changed on its own. The three cells below vary the ansatz, the encoding, and the observable, and reuse everything else untouched.

**A different ansatz.** `Ansatz.hardware_efficient` takes the rotation axes as an argument, so `("y", "z")` gives two rotations per qubit per layer — twice the parameters of `Ansatz.real_amplitudes`, and access to complex amplitudes rather than only real ones.

In [ ]:
he_params = hardware_efficient_size(num_qubits, layers=1, rotations=("y", "z"))


ansatz_he = Ansatz.hardware_efficient(num_qubits=num_qubits, layers=1, rotations=("y", "z"))


def circuit_he(x, w):
    qf = QuantumFloat(num_qubits)
    encoding.apply(qf, x)
    ansatz_he.apply(qf, w)
    return qf


def model_he(x, w):
    return expectation(lambda: circuit_he(x, w), Z(0), precision=0.05)()


@jaspify(terminal_sampling=True)
def train_he():
    loss = make_loss(model_he, X_train, y_train)
    # The losses are compared after the traced function returns: inside it they
    # are tracers, which have no truth value.
    return [
        minimize(loss, jnp.array([start] * he_params), method="COBYLA", options={"maxiter": 80}).fun
        for start in [0.5, 1.5, 2.5]
    ]


print(f"hardware_efficient(y,z), {he_params} parameters -> loss {min(float(f) for f in train_he()):.4f}")

**A different encoding.** `Encoding.dense_angle` packs *two* features per qubit — one as a rotation angle, one as a relative phase — so two qubits carry four features instead of two. The register stays the same size while the model takes twice the input.

In [ ]:
X_dense = jnp.array(
    [
        [0.0, 0.0, 0.1, 0.1],
        [0.2, 0.1, 0.0, 0.2],
        [np.pi, np.pi, 3.0, 3.0],
        [np.pi - 0.2, np.pi, 2.9, 3.1],
    ]
)


encoding_dense = Encoding.dense_angle(n_features=4)


def circuit_dense(x, w):
    qf = QuantumFloat(num_qubits)
    encoding_dense.apply(qf, x)
    ansatz.apply(qf, w)
    return qf


def model_dense(x, w):
    return expectation(lambda: circuit_dense(x, w), Z(0), precision=0.05)()


@jaspify(terminal_sampling=True)
def train_dense():
    loss = make_loss(model_dense, X_dense, y_train)
    return [
        minimize(loss, jnp.array([start] * num_params), method="COBYLA", options={"maxiter": 80}).fun
        for start in [0.5, 1.5, 2.5]
    ]


best_dense = min(float(f) for f in train_dense())
print(f"dense_angle_encoding, 4 features on {num_qubits} qubits -> loss {best_dense:.4f}")

**A different readout.** Passing a *list* of observables returns one entry per observable instead of a scalar. This is how a model with several outputs is read out — one per class, for instance.

A summed operator such as `Z(0) + Z(1)` would instead collapse both qubits into a single number, from which the individual contributions cannot be recovered.

In [ ]:
y_two = jnp.array([[-1.0, 1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, -1.0]])


def model_two(x, w):
    return expectation(lambda: circuit(x, w), [Z(0), Z(1)], precision=0.05)()


@jaspify(terminal_sampling=True)
def train_two():
    loss = make_loss(model_two, X_train, y_two)
    results = []
    for start in [0.5, 1.5, 2.5]:
        result = minimize(loss, jnp.array([start] * num_params), method="COBYLA", options={"maxiter": 80})
        results.append((result.fun, result.x))
    return results


loss_two, w_two = min(train_two(), key=lambda r: float(r[0]))
print(f"two observables -> loss {float(loss_two):.4f}")


@jaspify(terminal_sampling=True)
def predict_two():
    return jnp.array([model_two(X_train[i], w_two) for i in range(X_train.shape[0])])


print("predictions (one row per sample, one column per observable):")
print(np.round(np.asarray(predict_two()), 2))

## Amplitude encoding

`Encoding.amplitude` writes the features into the *amplitudes* of the register rather than into rotation angles. A register of $n$ qubits holds $2^n$ amplitudes, so it encodes exponentially more features per qubit than angle encoding — 4 features on 2 qubits, 1024 on 10.

Two consequences follow from a quantum state having unit norm. The feature vector is normalized internally, so only its *direction* is encoded and its length is lost. And the vector is zero-padded up to the size of the register, since a register holds a power of two amplitudes while feature vectors rarely do.

In [ ]:
qf = QuantumFloat(2)
Encoding.amplitude(n_features=3).apply(qf, [1.0, 2.0, 3.0])

# Padded to (1, 2, 3, 0), then normalized by sqrt(14).
print("amplitudes:", np.round(np.array(qf.qs.statevector("array")).real, 4))
print("expected  :", np.round(np.array([1, 2, 3, 0]) / np.sqrt(14), 4))

Note that `statevector("array")` is indexed in reverse bit order, so the amplitude of $\ket{i}$ is not at index $i$ of that array — which is why the two rows above agree only up to that reordering.

## A nonlinear problem

The clusters so far are linearly separable. XOR is not: the classes sit at opposite *corners*, and no single split of the input space separates them.

In [ ]:
X_xor = jnp.array([[0.0, 0.0], [0.0, np.pi], [np.pi, 0.0], [np.pi, np.pi]])
y_xor = jnp.array([1.0, -1.0, -1.0, 1.0])

for i in range(4):
    print(f"  features {np.round(np.asarray(X_xor[i]), 2)}  ->  label {float(y_xor[i]):+.0f}")

`Ansatz.data_reuploading` re-encodes the features between parameterized layers rather than only once at the start:

$$\prod_{l=1}^{L} W(w_l)\, S(x)$$

The reason is a spectral one. With a single encoding layer the circuit's output is a function of the features whose Fourier spectrum is fixed by the encoding, and no amount of added parameters widens it. With $L$ encoding layers the accessible spectrum grows with $L$, which is what lets a circuit on few qubits fit a richer function.

It takes an encoding and an ansatz, so any encoding can be alternated with any ansatz. The per-layer ansatz determines how the parameter vector is split into one block per layer, and `num_parameters` reports the total.

In [ ]:
reupload_layers = 3
reuploading = Ansatz.data_reuploading(
    num_qubits=num_qubits,
    encoding=encoding,
    ansatz=Ansatz.real_amplitudes(num_qubits=num_qubits, layers=1),
    layers=reupload_layers,
)
total_params = reuploading.num_parameters


def circuit_reupload(x, w):
    qf = QuantumFloat(num_qubits)
    reuploading.apply(qf, w, x)
    return qf


def model_reupload(x, w):
    return expectation(lambda: circuit_reupload(x, w), Z(0), precision=0.05)()


@jaspify(terminal_sampling=True)
def train_reupload():
    loss = make_loss(model_reupload, X_xor, y_xor)
    return [
        minimize(loss, jnp.array([start] * total_params), method="COBYLA", options={"maxiter": 150}).fun
        for start in [0.5, 1.5, 2.5]
    ]


print(f"data re-uploading, {reupload_layers} layers, {total_params} parameters")
print(f"XOR loss: {min(float(f) for f in train_reupload()):.4f}")

Whether this beats a single encoding layer on *this* problem depends heavily on the optimizer's starting point — with only four samples and a gradient-free local solver, the parameter count that re-uploading adds can cost as much in optimization difficulty as it gains in expressivity. The spectral argument is about what the circuit *can* represent, not about what COBYLA will find in 150 iterations. On a larger problem, with more data and a better-conditioned optimizer, the gain is easier to realise.

## A different optimizer

`minimize` also offers SPSA, which estimates a descent direction from two evaluations per iteration regardless of the parameter count. That makes it cheaper than COBYLA on large parameter vectors, at the cost of a noisier path.

In [ ]:
@jaspify(terminal_sampling=True)
def train_spsa():
    loss = make_loss(model, X_train, y_train)
    w0 = jnp.array([1.5] * num_params)
    result = minimize(loss, w0, method="SPSA", options={"maxiter": 60})
    return loss(w0), result.fun


before, after = train_spsa()
print(f"SPSA: loss {float(before):.3f} -> {float(after):.3f}")

## Hybrid models: classical layers around the quantum block

Everything so far has been a quantum circuit with classical data going in. A **hybrid** model puts trainable classical layers on *both* sides of it:

```
x  ->  classical  ->  quantum block  ->  classical  ->  prediction
       (jnp ops)      (encoding +        (jnp ops)
                       ansatz)
```

This works because `expectation` returns an ordinary function of arrays. Inside `@jaspify`, `jnp` operations compose freely with quantum code, so the classical parts are just arithmetic.

Two things make it practical. First, ask for **one observable per output** you want to feed forward — a list of observables gives a vector rather than a single number. Second, the parameters now come in *named groups*: a classical layer in front, the ansatz parameters, a classical layer behind.

In [ ]:
num_qubits = 2
layers = 1

# The first feature decides the label, so this task is separable — the point
# here is the wiring, not the difficulty.
X_train = jnp.array([[0.0, 0.0], [0.0, np.pi], [np.pi, 0.0], [np.pi, np.pi]])
y_train = jnp.array([1.0, 1.0, -1.0, -1.0])

encoding = Encoding.angle(n_features=num_qubits)
ansatz = Ansatz.real_amplitudes(num_qubits=num_qubits, layers=layers)


def circuit(x, w):
    qf = QuantumFloat(num_qubits)
    encoding.apply(qf, x)
    ansatz.apply(qf, w)
    return qf


# Two observables -> a length-2 vector for the classical layer to consume.
quantum_block = expectation(circuit, [Z(0), Z(1)], precision=0.05)


def hybrid_model(x, p):
    h = jnp.tanh(x * p["w_in"] + p["b_in"])          # classical, before
    q = quantum_block(h, p["theta"])                 # quantum
    return jnp.tanh(jnp.dot(q, p["w_out"]) + p["b_out"])   # classical, after

### Named parameters and the flat vector

`minimize` takes a single flat vector, but the model above wants a dictionary. Converting between the two by hand — slicing the vector with computed offsets — is a well-known source of silent error: an off-by-one hands the wrong numbers to the wrong layer and nothing raises, the model simply trains on nonsense.

Jax already solves this. `ravel_pytree` flattens any nested structure of arrays into one vector and hands back an `unravel` function that restores the exact shapes, including scalars. The `unravel` works **inside** the traced function, which is where the optimizer needs it.

In [ ]:
from jax.flatten_util import ravel_pytree

parameters = {
    "w_in": jnp.ones(num_qubits),
    "b_in": jnp.zeros(num_qubits),
    "theta": jnp.full(ansatz.num_parameters, 0.5),
    "w_out": jnp.ones(num_qubits),
    "b_out": jnp.array(0.0),
}

flat0, unravel = ravel_pytree(parameters)

print("groups     :", {k: tuple(np.shape(v)) for k, v in parameters.items()})
print("flat vector:", flat0.shape)
print("roundtrip  :", sorted(unravel(flat0)))

The loss unravels the vector it is given and then calls the model normally. That is the whole trick — the optimizer never sees the structure, and the model never sees the flat vector.

In [ ]:
def make_hybrid_loss(X, y):
    def loss(v):
        p = unravel(v)
        total = 0.0
        for i in range(X.shape[0]):
            total = total + (hybrid_model(X[i], p) - y[i]) ** 2
        return total / X.shape[0]

    return loss


@jaspify(terminal_sampling=True)
def train_hybrid():
    loss = make_hybrid_loss(X_train, y_train)

    results = []
    for scale in [0.5, 1.5]:
        v0 = flat0 * scale
        result = minimize(loss, v0, method="COBYLA", options={"maxiter": 60})
        results.append((scale, loss(v0), result.fun, result.x))
    return results


results = train_hybrid()

for scale, before, after, _ in results:
    print(f"start x{scale}:  loss {float(before):.3f} -> {float(after):.3f}")

best = min(results, key=lambda r: float(r[2]))
print(f"\nbest final loss: {float(best[2]):.4f}")

Reading the trained parameters back out is the same call in reverse, and both halves of the model have moved.

In [ ]:
p_opt = unravel(jnp.asarray(best[3]))

print("recovered groups:", sorted(p_opt))
print("quantum   moved :", not np.allclose(p_opt["theta"], parameters["theta"]))
print("classical moved :", not np.allclose(p_opt["w_out"], parameters["w_out"]))

### What this does and does not give you

The composition is real: classical layers train alongside the quantum ones, through one optimizer call, and the same code runs traced.

What it is **not** is end-to-end differentiation. `minimize` is gradient-free, so the classical layers are optimized by sampling the loss rather than by backpropagation — workable for the small parameter counts here, but it does not scale the way a classical network does, and it is why this section keeps the classical layers tiny.

Note also the cost: with a vector readout, each expectation value is estimated from its own sampling run, so *m* observables cost *m* runs even when one would do. A wider readout gets expensive quickly.

## Summary

The loss was a function and training was a call to `minimize`. The encoding and the ansatz are configured objects, each independent of the others, which is why every stage could be swapped without touching the rest:

| Stage | Primitives |
|---|---|
| Preprocessing | `minmax_scale`, `normalize`, `binarize` |
| Encoding | `Encoding.angle`, `Encoding.dense_angle`, `Encoding.amplitude`, `Encoding.basis` |
| Ansatz | `Ansatz.real_amplitudes`, `Ansatz.hardware_efficient`, `Ansatz.data_reuploading` |
| Measurement | `expectation` with one or several observables |
| Optimizer | `minimize` with `COBYLA` or `SPSA` |

The preprocessing functions exist because each encoding has an input contract: bounded angles for angle encoding, unit norm for amplitude encoding, strict bits for basis encoding. Anything beyond that — imputation, dimensionality reduction, categorical features — is ordinary classical data preparation, and the established tools do it better.

Three things are worth carrying forward:

**Features must be scaled before encoding.** Rotation angles are $2\pi$-periodic, and nothing warns you when a feature wraps around — the circuit runs and the model simply trains badly. `minmax_scale` fits on the training data and reuses that range everywhere else.

**The optimizers are local.** Results depend on the starting point, so vary it before concluding anything about a model.

**Tracing is not differentiation.** Jasp makes the circuit fast to run; it does not provide gradients through quantum execution. That is why training is gradient-free here.